# 05 — Full Evaluation
Confusion Matrix, ROC, Kappa, MCC, Log Loss, Learning Curves

**Optimized:** tqdm, subsampled learning curves, batch predictions

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, pickle, os
from tqdm.notebook import tqdm
if os.path.basename(os.getcwd())=='notebooks': os.chdir('..')
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (accuracy_score,f1_score,cohen_kappa_score,matthews_corrcoef,
    roc_auc_score,log_loss,confusion_matrix,classification_report,roc_curve,auc,fbeta_score)
from sklearn.preprocessing import label_binarize
SEED=42; PROC='data/processed'; MOD='models'; CH='charts'
os.makedirs(f'{CH}/evaluation',exist_ok=True)
os.makedirs(f'{CH}/model_comparison',exist_ok=True)
print('Ready')

In [ ]:
X=pd.read_csv(f'{PROC}/X_processed.csv')
y=pd.read_csv(f'{PROC}/y_target.csv').iloc[:,0]
CN=pickle.load(open(f'{MOD}/class_names.pkl','rb'))
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=SEED,stratify=y)
# Load all trained models
zoo_dir=f'{MOD}/model_zoo'
all_models={}
for f in sorted(os.listdir(zoo_dir)):
    if f.endswith('.pkl'):
        name=f.replace('.pkl','')
        all_models[name]=pickle.load(open(f'{zoo_dir}/{f}','rb'))
print(f'Loaded {len(all_models)} models')

## 1. Evaluate All Models (tqdm)

In [ ]:
def full_metrics(y_true,y_pred,y_proba,cn):
    m={'accuracy':accuracy_score(y_true,y_pred),
       'f1_macro':f1_score(y_true,y_pred,average='macro',zero_division=0),
       'f1_weighted':f1_score(y_true,y_pred,average='weighted',zero_division=0),
       'f2_macro':fbeta_score(y_true,y_pred,beta=2,average='macro',zero_division=0),
       'cohens_kappa':cohen_kappa_score(y_true,y_pred),
       'mcc':matthews_corrcoef(y_true,y_pred)}
    if y_proba is not None:
        try: m['log_loss']=log_loss(y_true,y_proba)
        except: m['log_loss']=np.nan
        try:
            yb=label_binarize(y_true,classes=list(range(len(cn))))
            m['roc_auc_macro']=roc_auc_score(yb,y_proba,multi_class='ovr',average='macro')
            m['roc_auc_weighted']=roc_auc_score(yb,y_proba,multi_class='ovr',average='weighted')
        except: m['roc_auc_macro']=np.nan; m['roc_auc_weighted']=np.nan
    return m

results={}; preds_store={}
for name,model in tqdm(all_models.items(),desc='Evaluating Models'):
    try:
        yp=model.predict(X_test)
        ypr=None
        if hasattr(model,'predict_proba'):
            try: ypr=model.predict_proba(X_test)
            except: pass
        results[name]=full_metrics(y_test.values,yp,ypr,CN)
        preds_store[name]={'pred':yp,'proba':ypr}
        tqdm.write(f'  ✓ {name}: F1={results[name]["f1_macro"]:.4f}')
    except Exception as e:
        tqdm.write(f'  ✗ {name}: {e}')

res_df=pd.DataFrame(results).T.sort_values('f1_macro',ascending=False)
res_df.index.name='Model'; res_df.reset_index(inplace=True)
res_df.to_csv(f'{MOD}/all_results.csv',index=False)
res_df.style.background_gradient(subset=['f1_macro','cohens_kappa','mcc','roc_auc_macro'],cmap='Greens')

## 2. Comparison Charts

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(20,7))
for i,met in enumerate(['accuracy','f1_macro','cohens_kappa']):
    d=res_df.sort_values(met)
    axes[i].barh(d['Model'],d[met],color=plt.cm.viridis(np.linspace(0.2,0.8,len(d))),edgecolor='black')
    axes[i].set_title(met.replace('_',' ').title(),fontweight='bold')
    axes[i].set_xlim(0,1)
fig.suptitle('All Models Compared',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig(f'{CH}/model_comparison/08_all_comparison.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. Confusion Matrix — Champion

In [ ]:
champ=res_df.iloc[0]['Model']
print(f'Champion: {champ}')
yp=preds_store[champ]['pred']
cm=confusion_matrix(y_test,yp)
fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=CN,yticklabels=CN,ax=a1)
a1.set_title('Raw Counts'); a1.set_xlabel('Predicted'); a1.set_ylabel('Actual')
cmn=cm.astype(float)/cm.sum(axis=1,keepdims=True)
sns.heatmap(cmn,annot=True,fmt='.2f',cmap='Oranges',xticklabels=CN,yticklabels=CN,ax=a2)
a2.set_title('Normalized'); a2.set_xlabel('Predicted'); a2.set_ylabel('Actual')
fig.suptitle(f'Confusion Matrix: {champ}',fontweight='bold')
plt.tight_layout(); plt.savefig(f'{CH}/evaluation/09_confusion_{champ}.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. ROC Curves (Multi-class)

In [ ]:
ypr=preds_store[champ]['proba']
if ypr is not None:
    yb=label_binarize(y_test.values,classes=list(range(len(CN))))
    fig,ax=plt.subplots(figsize=(8,6))
    clrs=['#2196F3','#4CAF50','#FF9800','#E91E63']
    for i,name in enumerate(CN):
        fpr,tpr,_=roc_curve(yb[:,i],ypr[:,i])
        ax.plot(fpr,tpr,color=clrs[i],lw=2,label=f'{name} (AUC={auc(fpr,tpr):.3f})')
    ax.plot([0,1],[0,1],'k--',alpha=0.5)
    ax.set_title(f'Multi-Class ROC: {champ}',fontweight='bold')
    ax.legend(loc='lower right')
    plt.savefig(f'{CH}/evaluation/10_roc_{champ}.png',dpi=150,bbox_inches='tight'); plt.show()
else: print('No probabilities available')

## 5. Learning Curves (Optimized: 20k subsample, 5 sizes)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
# Subsample for learning curves
LC_SAMPLE=20000
idx_lc=np.random.RandomState(SEED).choice(len(X_train),min(LC_SAMPLE,len(X_train)),replace=False)
X_lc=X_train.iloc[idx_lc]; y_lc=y_train.iloc[idx_lc]
print(f'Learning curve subsample: {len(X_lc):,}')

rf=RandomForestClassifier(n_estimators=100,max_depth=10,class_weight='balanced',random_state=SEED,n_jobs=-1)
ts,train_s,val_s=learning_curve(rf,X_lc,y_lc,cv=3,scoring='f1_macro',
    train_sizes=np.linspace(0.2,1.0,5),n_jobs=-1,random_state=SEED)
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(ts,train_s.mean(1),'o-',label='Train',color='#2196F3')
ax.fill_between(ts,train_s.mean(1)-train_s.std(1),train_s.mean(1)+train_s.std(1),alpha=0.15,color='#2196F3')
ax.plot(ts,val_s.mean(1),'o-',label='Validation',color='#4CAF50')
ax.fill_between(ts,val_s.mean(1)-val_s.std(1),val_s.mean(1)+val_s.std(1),alpha=0.15,color='#4CAF50')
ax.set_xlabel('Training Size'); ax.set_ylabel('F1 Macro')
ax.set_title('Learning Curve (Random Forest)',fontweight='bold'); ax.legend()
plt.savefig(f'{CH}/evaluation/11_learning_curve.png',dpi=150,bbox_inches='tight'); plt.show()

## 6. Validation Curves (Optimized: 5 depths)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import validation_curve
# Subsample for validation curves
VC_SAMPLE=20000
idx_vc=np.random.RandomState(SEED+1).choice(len(X_train),min(VC_SAMPLE,len(X_train)),replace=False)
X_vc=X_train.iloc[idx_vc]; y_vc=y_train.iloc[idx_vc]

dt=DecisionTreeClassifier(class_weight='balanced',random_state=SEED)
pr=[2,5,10,15,20]
train_s2,val_s2=validation_curve(dt,X_vc,y_vc,param_name='max_depth',
    param_range=pr,cv=3,scoring='f1_macro',n_jobs=-1)
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(pr,train_s2.mean(1),'o-',label='Train',color='#2196F3')
ax.plot(pr,val_s2.mean(1),'o-',label='Validation',color='#4CAF50')
ax.fill_between(pr,train_s2.mean(1)-train_s2.std(1),train_s2.mean(1)+train_s2.std(1),alpha=0.15)
ax.fill_between(pr,val_s2.mean(1)-val_s2.std(1),val_s2.mean(1)+val_s2.std(1),alpha=0.15)
ax.set_xlabel('max_depth'); ax.set_ylabel('F1 Macro')
ax.set_title('Validation Curve (Decision Tree)',fontweight='bold'); ax.legend()
plt.savefig(f'{CH}/evaluation/12_validation_curve.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
print(f'\n=== CHAMPION: {champ} ===')
r=results[champ]
for k,v in r.items():
    if isinstance(v,(int,float,np.floating)): print(f'  {k}: {v:.4f}')
print('\nNotebook 05 complete')